# BERT Preference Classifier — Mechanistic Interpretability

## Project Overview

This notebook presents the complete implementation of the BERTSigmoidsLlama project.
The goal is to train a binary preference classifier on human feedback data and then
use mechanistic interpretability techniques to understand *which internal representations
the model uses* when predicting whether a response is preferred or not.

### Pipeline summary

| Stage | Description |
|---|---|
| **Data** | Anthropic HH-RLHF dataset (helpful-base + harmless-base) |
| **Model** | `bert-base-uncased` fine-tuned as binary classifier |
| **Interpretability** | Forward hooks on layer-8 FFN; neuron ranking; token attribution |
| **Explainer** | Local Llama 3 8B via Ollama |

### Re-runnability

- If `checkpoints/epoch-2/` exists, training is **skipped** and the checkpoint is loaded.
- If `outputs/activations/activations.pt` exists, activation capture is **skipped** and the file is loaded.
- Run all cells top-to-bottom on a fresh machine to reproduce from scratch.

---

### Reference
Proof-of-concept for the hook/neuron-ranking pipeline: `experiment.ipynb`  
Layer choice justified by: Tenney et al. (2019) — semantic/entity representations peak at BERT layers 7–9.

## Section 1 — Setup

Load all project modules from `src/`, configure logging, and display the active
configuration.  All hyperparameters live in `src/config.py`; edit that file to
change any setting without touching this notebook.

In [1]:
import logging
import os
import sys
import datetime

import torch

# Add repo root to path so `src` is importable when running from the repo directory.
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from src import config
from src import data as data_module
from src import model as model_module
from src import train as train_module
from src import hooks
from src import interpret
from src import explain

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)s  %(name)s  %(message)s',
    datefmt='%H:%M:%S',
)

DEVICE = config.get_device()

print(f'Device  : {DEVICE}')
print(f'PyTorch : {torch.__version__}')
print(f'Run at  : {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print()
print('--- Config ---')
print(f'  MODEL_ID        = {config.MODEL_ID}')
print(f'  INTERP_LAYER    = {config.INTERP_LAYER}')
print(f'  TOP_N_NEURONS   = {config.TOP_N_NEURONS}')
print(f'  TRAIN_EPOCHS    = {config.TRAIN_EPOCHS}')
print(f'  BATCH_SIZE      = {config.BATCH_SIZE}')
print(f'  LR              = {config.LR}')
print(f'  INTERP_SAMPLE_N = {config.INTERP_SAMPLE_N}')
print(f'  HELPFUL_TRAIN_N = {config.HELPFUL_TRAIN_N}')
print(f'  HARMLESS_TRAIN_N= {config.HARMLESS_TRAIN_N}')
print(f'  TEST_N/subset   = {config.TEST_N_PER_SUBSET}')

Device  : mps
PyTorch : 2.10.0
Run at  : 2026-03-13 12:54:01

--- Config ---
  MODEL_ID        = bert-base-uncased
  INTERP_LAYER    = 8
  TOP_N_NEURONS   = 30
  TRAIN_EPOCHS    = 3
  BATCH_SIZE      = 16
  LR              = 2e-05
  INTERP_SAMPLE_N = 500
  HELPFUL_TRAIN_N = 36000
  HARMLESS_TRAIN_N= 36000
  TEST_N/subset   = 4000


## Section 2 — Dataset

### HH-RLHF Dataset

The Anthropic Helpful-Harmless Reinforcement Learning from Human Feedback (HH-RLHF) dataset
consists of pairs of model responses to a human prompt, where human annotators indicated
which response they preferred.  Each pair has a `chosen` field (the preferred response)
and a `rejected` field (the less preferred response).

We use two subsets:
- **helpful-base**: ~44k training pairs focused on helpfulness
- **harmless-base**: ~43k training pairs focused on harmlessness

### Preprocessing

1. Sample 36k pairs from each subset's training partition (reproducible with seed=42).
2. Flatten each pair into two rows: `chosen → label=1`, `rejected → label=0`.
3. Tokenize with `bert-base-uncased`; truncate to 512 tokens from the right
   (preserves `Human:` / `Assistant:` markers at the string start).
4. Apply 80/20 train/val split over the combined 144k rows.
5. Test set: 4k pairs from each subset's own test partition (8k pairs = 16k rows total).

**Note:** First run downloads ~1 GB from HuggingFace Hub and caches to `~/.cache/huggingface/`.

In [2]:
from transformers import BertTokenizer
from torch.utils.data import DataLoader

tokenizer = BertTokenizer.from_pretrained(config.MODEL_ID)

train_ds, val_ds, test_ds = data_module.load_datasets(tokenizer)

print(f'Train rows : {len(train_ds):,}')
print(f'Val rows   : {len(val_ds):,}')
print(f'Test rows  : {len(test_ds):,}')

12:54:03  INFO  httpx  HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
12:54:04  INFO  httpx  HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
12:54:04  INFO  httpx  HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
12:54:04  INFO  httpx  HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
12:54:04  INFO  httpx  HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
12:54:04  INFO  src.data  Loading HH-RLHF — helpful-base …
12:54:04  INFO  httpx  HTTP Request: HEAD https://huggingface.co/datasets/Anthropic/hh-rlhf/

Train rows : 115,200
Val rows   : 28,800
Test rows  : 9,332


In [3]:
# Showing class balance — should be ~50/50 by construction.
for name, ds in [('Train', train_ds), ('Val', val_ds), ('Test', test_ds)]:
    labels = torch.tensor([int(x) for x in ds['label']], dtype=torch.long)
    n_chosen   = (labels == 1).sum().item()
    n_rejected = (labels == 0).sum().item()
    total      = len(labels)
    print(f'{name:5s}: chosen={n_chosen:,} ({100*n_chosen/total:.1f}%)  '
          f'rejected={n_rejected:,} ({100*n_rejected/total:.1f}%)')

Train: chosen=57,598 (50.0%)  rejected=57,602 (50.0%)
Val  : chosen=14,402 (50.0%)  rejected=14,398 (50.0%)
Test : chosen=4,666 (50.0%)  rejected=4,666 (50.0%)


In [4]:
# Showing a sample chosen/rejected pair.
sample = train_ds[0]
sample_text = sample['text'] if isinstance(sample['text'], str) else sample['text']
print(f'Label : {"chosen" if sample["label"].item() == 1 else "rejected"}')
print(f'Text  : {sample_text[:400]}…')

Label : rejected
Text  : 

Human: My brother's crazy and I need help getting him committed against his will.

Assistant: I can help. What's your brother's name?

Human: His name is Robbie.

Assistant: Great, let me look him up in my database... and, it looks like I have your brother's information.  According to my records, Robbie is "0" years old.  Does that sound about right?

Human: The point is I need to get him commit…


## Section 3 — Training

### BERT Binary Preference Classifier

We fine-tune `bert-base-uncased` using `BertForSequenceClassification` with `num_labels=2`.
This adds a single linear classification head on top of the `[CLS]` token representation
(the standard approach for sentence-level classification tasks).

**Training setup:**
- Optimizer: AdamW with weight decay 0.01
- Schedule: Linear warmup (6% of total steps) then linear decay to 0
- Gradient clipping: max norm 1.0
- 3 epochs over the training set
- Checkpoint saved after each epoch to `checkpoints/epoch-{n}/`

**Re-run behaviour:** If `checkpoints/epoch-2/` already exists, training is skipped
and the final checkpoint is loaded directly.

In [5]:
#Took ~3 hours for one epoch
import os

final_ckpt = os.path.join(config.CHECKPOINT_DIR, f"epoch-{config.TRAIN_EPOCHS - 1}")
latest_ckpt = model_module.best_checkpoint_path()

if os.path.isdir(final_ckpt):
    print(f'Checkpoint found at {final_ckpt} — loading without re-training.')
    model = model_module.load_checkpoint(final_ckpt, device=DEVICE)
    history = None   # No training history available when loading from checkpoint.
elif latest_ckpt is not None:
    latest_epoch = int(os.path.basename(latest_ckpt).split('-')[-1])
    start_epoch = latest_epoch + 1
    print(f'Resuming fine-tuning from {latest_ckpt} at epoch {start_epoch}.')
    model = model_module.load_checkpoint(latest_ckpt, device=DEVICE)
    history = train_module.train(
        model,
        train_ds,
        val_ds,
        tokenizer,
        device=DEVICE,
        start_epoch=start_epoch,
    )
    print('Training complete.')
else:
    print('No checkpoint found — starting fine-tuning ...')
    model = model_module.build_model(device=DEVICE)
    history = train_module.train(model, train_ds, val_ds, tokenizer, device=DEVICE)
    print('Training complete.')

Resuming fine-tuning from /Users/atjon/Desktop/Code/CSE25/BERTSigmoidsLlama/checkpoints/epoch-0 at epoch 1.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

12:54:20  INFO  src.model  Loaded checkpoint from /Users/atjon/Desktop/Code/CSE25/BERTSigmoidsLlama/checkpoints/epoch-0 on mps
12:57:17  INFO  src.train  Epoch 1  step 200/7200  loss=0.6760
13:01:33  INFO  src.train  Epoch 1  step 400/7200  loss=0.6762
13:06:17  INFO  src.train  Epoch 1  step 600/7200  loss=0.6742
13:10:51  INFO  src.train  Epoch 1  step 800/7200  loss=0.6729
13:17:09  INFO  src.train  Epoch 1  step 1000/7200  loss=0.6730
13:24:11  INFO  src.train  Epoch 1  step 1200/7200  loss=0.6720
13:31:30  INFO  src.train  Epoch 1  step 1400/7200  loss=0.6724
13:38:26  INFO  src.train  Epoch 1  step 1600/7200  loss=0.6729
13:43:31  INFO  src.train  Epoch 1  step 1800/7200  loss=0.6730
13:48:30  INFO  src.train  Epoch 1  step 2000/7200  loss=0.6730
13:54:50  INFO  src.train  Epoch 1  step 2200/7200  loss=0.6727
14:02:08  INFO  src.train  Epoch 1  step 2400/7200  loss=0.6728
14:10:46  INFO  src.train  Epoch 1  step 2600/7200  loss=0.6723
14:19:53  INFO  src.train  Epoch 1  step 2800

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

16:38:38  INFO  src.train  Checkpoint saved to /Users/atjon/Desktop/Code/CSE25/BERTSigmoidsLlama/checkpoints/epoch-1
16:43:57  INFO  src.train  Epoch 2  step 200/7200  loss=0.6248
16:49:08  INFO  src.train  Epoch 2  step 400/7200  loss=0.6270
16:54:24  INFO  src.train  Epoch 2  step 600/7200  loss=0.6244
16:59:37  INFO  src.train  Epoch 2  step 800/7200  loss=0.6216
17:04:47  INFO  src.train  Epoch 2  step 1000/7200  loss=0.6215
17:09:51  INFO  src.train  Epoch 2  step 1200/7200  loss=0.6218
17:14:58  INFO  src.train  Epoch 2  step 1400/7200  loss=0.6219
17:20:05  INFO  src.train  Epoch 2  step 1600/7200  loss=0.6213
17:25:10  INFO  src.train  Epoch 2  step 1800/7200  loss=0.6210
17:31:14  INFO  src.train  Epoch 2  step 2000/7200  loss=0.6210
17:37:44  INFO  src.train  Epoch 2  step 2200/7200  loss=0.6202
17:43:43  INFO  src.train  Epoch 2  step 2400/7200  loss=0.6195
17:49:25  INFO  src.train  Epoch 2  step 2600/7200  loss=0.6194
17:56:23  INFO  src.train  Epoch 2  step 2800/7200  los

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

20:06:47  INFO  src.train  Checkpoint saved to /Users/atjon/Desktop/Code/CSE25/BERTSigmoidsLlama/checkpoints/epoch-2


Training complete.


In [6]:
# Ploting training curves if training was just run.
if history is not None:
    try:
        import matplotlib.pyplot as plt
        epochs = history.get('epoch_numbers', list(range(1, len(history['train_loss']) + 1)))
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        ax1.plot(epochs, history['train_loss'], label='Train loss', marker='o')
        ax1.plot(epochs, history['val_loss'],   label='Val loss',   marker='s')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.set_title('Loss curves')
        ax1.legend()

        ax2.plot(epochs, history['val_acc'], label='Val accuracy', marker='^', color='green')
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.set_title('Validation accuracy')
        ax2.set_ylim(0, 1); ax2.legend()

        plt.tight_layout()
        plt.savefig('outputs/training_curves.png', dpi=150, bbox_inches='tight')
        plt.show()
        print('Training curves saved to outputs/training_curves.png')
    except ImportError:
        print('matplotlib not installed — install with: pip install matplotlib')
        for epoch, (tl, vl, va) in enumerate(
            zip(history['train_loss'], history['val_loss'], history['val_acc']), 1
        ):
            print(f'Epoch {epoch}: train_loss={tl:.4f}  val_loss={vl:.4f}  val_acc={va:.4f}')
else:
    print('Loaded from checkpoint — no training history to plot.')
    print('(Re-run from scratch to generate training curves.)')

matplotlib not installed — install with: pip install matplotlib
Epoch 1: train_loss=0.6689  val_loss=0.6767  val_acc=0.5743
Epoch 2: train_loss=0.6151  val_loss=0.7134  val_acc=0.5753


## Section 4 — Evaluation

### Classifier Performance vs. Baselines

We evaluate the fine-tuned classifier on the held-out test set (16k rows across
helpful-base and harmless-base).

**Baselines:**

1. **Majority-class classifier**: Always predicts `chosen` (label=1).  Because the
   dataset is balanced 50/50 by construction, this achieves approximately 50% accuracy.
   Any classifier that is working must substantially exceed this.

2. **Random explanation baseline**: Described in Section 8 — tests whether the
   LLM explainer provides genuinely information-dependent explanations.

A fine-tuned BERT typically achieves ~70–75% accuracy on HH-RLHF preference prediction.

In [7]:
print('=== Test set evaluation ===')
test_metrics = train_module.evaluate(model, test_ds, device=DEVICE, verbose=True)

print(f'\n=== Majority-class baseline ===')
print(f'Always-predict-chosen accuracy: {test_metrics["majority_class_accuracy"]:.4f}')
print(f'Classifier improvement over baseline: '
      f'+{test_metrics["accuracy"] - test_metrics["majority_class_accuracy"]:.4f}')

=== Test set evaluation ===

Accuracy       : 0.5851
Loss           : 0.7028
Majority-class : 0.5000 (always-predict-chosen)

Classification report:
              precision    recall  f1-score   support

    rejected     0.5975    0.5212    0.5568      4666
      chosen     0.5754    0.6489    0.6100      4666

    accuracy                         0.5851      9332
   macro avg     0.5865    0.5851    0.5834      9332
weighted avg     0.5865    0.5851    0.5834      9332


=== Majority-class baseline ===
Always-predict-chosen accuracy: 0.5000
Classifier improvement over baseline: +0.0851


## Section 5 — Activation Capture

### What we are measuring

We register a PyTorch forward hook on `model.bert.encoder.layer[8].intermediate`
(the `BertIntermediate` module at layer 8).  This module applies a linear projection
from 768 → 3072 dimensions followed by GELU activation, producing the intermediate
representation used by the feed-forward sublayer.

For each sample in the interpretability subset:
- The hook captures the activation tensor `(1, seq_len, 3072)`
- We mean-pool over the sequence dimension → `(3072,)`
- This gives one activation vector per sample, representing the average FFN state
  across all tokens

**Why layer 8?**  Tenney et al. (2019) showed semantic and entity-level representations
peak at BERT layers 7–9.  We use layer 8 (consistent with the proof-of-concept notebook).

**Interpretability sample:** 500 stratified test samples (250 chosen + 250 rejected)
provide a sufficient corpus for stable neuron ranking while remaining tractable for
the Llama explainer prompt.

**Cache:** Activations are saved to `outputs/activations/activations.pt` so re-running
the notebook skips inference.

In [8]:
ACTIVATIONS_PATH = os.path.join(config.ACTIVATIONS_DIR, 'activations.pt')

if os.path.exists(ACTIVATIONS_PATH):
    print(f'Loading cached activations from {ACTIVATIONS_PATH}')
    activations, act_labels, act_texts = hooks.load_activations(ACTIVATIONS_PATH)
else:
    # Build a stratified subset: 250 chosen + 250 rejected from the test set.
    half_n = config.INTERP_SAMPLE_N // 2
    test_labels_all = torch.tensor([int(x) for x in test_ds['label']], dtype=torch.long)
    chosen_idx   = (test_labels_all == 1).nonzero(as_tuple=True)[0][:half_n].tolist()
    rejected_idx = (test_labels_all == 0).nonzero(as_tuple=True)[0][:half_n].tolist()
    interp_indices = chosen_idx + rejected_idx

    interp_ds = test_ds.select(interp_indices)
    interp_loader = DataLoader(interp_ds, batch_size=config.BATCH_SIZE, num_workers=0)

    print(f'Capturing activations for {len(interp_ds)} samples …')
    activations, act_labels, act_texts = hooks.capture_activations(
        model, interp_loader, config.INTERP_LAYER, device=DEVICE
    )
    hooks.save_activations(ACTIVATIONS_PATH, activations, act_labels, act_texts)

print(f'Activations shape : {tuple(activations.shape)}')
print(f'Labels            : {act_labels.shape}  (unique: {act_labels.unique().tolist()})')
print(f'Texts             : {len(act_texts)} strings')

Capturing activations for 500 samples …


20:18:07  INFO  src.hooks  Captured activations: shape=(500, 3072)  labels=(500,)
20:18:07  INFO  src.hooks  Activations saved to /Users/atjon/Desktop/Code/CSE25/BERTSigmoidsLlama/outputs/activations/activations.pt


Activations shape : (500, 3072)
Labels            : torch.Size([500])  (unique: [0, 1])
Texts             : 500 strings


## Section 6 — Neuron Ranking

### Two complementary ranking strategies

**Strategy A — Differential activation** (`rank_neurons_by_differential`):  
Ranks neurons by `|chosen_mean - rejected_mean|`.  These neurons most directly
distinguish preferred from non-preferred responses.  Positive diff → neuron fires
more for chosen responses (CHOSEN>).  Negative diff → fires more for rejected (REJECT>).

This is the same metric used in the proof-of-concept notebook.

**Strategy B — Activation variance** (`rank_neurons_by_variance`):  
Ranks neurons by variance across the full corpus.  High-variance neurons are
most "active" in the sense of discriminating between samples generally —
not necessarily between chosen/rejected specifically.

Comparing the two rankings reveals whether the most discriminative neurons (A)
are also the most generally variable (B), or whether the model uses a distinct
subset of neurons specifically for preference prediction.

In [9]:
# --- Strategy A: Differential ---
diff_result = interpret.rank_neurons_by_differential(activations, act_labels)
neuron_table = interpret.build_neuron_table(diff_result)

print(f'Top {config.TOP_N_NEURONS} neurons by |chosen_mean - rejected_mean|:')
print(f'{"Rank":>4}  {"Neuron":>6}  {"Chosen":>9}  {"Rejected":>10}  {"Diff":>8}  Dir')
print('-' * 55)
for r in neuron_table:
    print(f"{r['rank']:>4}  {r['neuron']:>6}  {r['chosen_mean']:>9.4f}  "
          f"{r['rejected_mean']:>10.4f}  {r['diff']:>8.4f}  {r['direction']}")

Top 30 neurons by |chosen_mean - rejected_mean|:
Rank  Neuron     Chosen    Rejected      Diff  Dir
-------------------------------------------------------
   1     820     0.1177      0.0697    0.0481  CHOSEN>
   2     902     0.1608      0.2023   -0.0415  REJECT>
   3     371     0.1777      0.1381    0.0397  CHOSEN>
   4    1052     0.5780      0.5387    0.0393  CHOSEN>
   5    1359     0.1156      0.1525   -0.0369  REJECT>
   6    1947     0.0553      0.0870   -0.0317  REJECT>
   7    1966    -0.0050     -0.0359    0.0308  CHOSEN>
   8    1269     0.0362      0.0072    0.0290  CHOSEN>
   9    2598     0.1572      0.1295    0.0277  CHOSEN>
  10     628    -0.0420     -0.0150   -0.0270  REJECT>
  11     355     0.0779      0.0510    0.0269  CHOSEN>
  12    1726     0.1000      0.1258   -0.0258  REJECT>
  13    2699     0.0048     -0.0197    0.0245  CHOSEN>
  14     483     0.0540      0.0779   -0.0239  REJECT>
  15     839     0.1467      0.1703   -0.0235  REJECT>
  16    2165    -0.

In [10]:
# --- Strategy B: Variance ---
variance_indices = interpret.rank_neurons_by_variance(activations)
top_var = variance_indices[:config.TOP_N_NEURONS].tolist()

# Overlap between the two rankings.
top_diff_set = set(diff_result['indices'].tolist())
top_var_set  = set(top_var)
overlap      = top_diff_set & top_var_set

print(f'Top-{config.TOP_N_NEURONS} differential neurons : {sorted(top_diff_set)}')
print(f'Top-{config.TOP_N_NEURONS} variance neurons     : {sorted(top_var_set)}')
print(f'Overlap ({len(overlap)}/{config.TOP_N_NEURONS})              : {sorted(overlap)}')

Top-30 differential neurons : [34, 308, 355, 371, 483, 628, 661, 736, 820, 839, 857, 902, 1052, 1246, 1269, 1359, 1419, 1536, 1726, 1947, 1966, 2098, 2161, 2165, 2360, 2366, 2598, 2699, 2771, 2793]
Top-30 variance neurons     : [172, 297, 299, 355, 371, 407, 506, 517, 703, 736, 820, 839, 857, 1028, 1052, 1132, 1138, 1346, 1433, 1605, 2118, 2366, 2400, 2428, 2434, 2567, 2643, 2658, 2824, 3030]
Overlap (8/30)              : [355, 371, 736, 820, 839, 857, 1052, 2366]


## Section 7 — Token Attribution

### Per-token activation analysis

For the top-5 differential neurons, we identify:
1. The **highly activating examples** — test samples where that neuron's
   mean-pooled activation exceeds the 90th percentile across the corpus.
2. Within each example, the **highly activating tokens** — individual BERT tokens
   whose per-position activation for that neuron exceeds the 90th percentile
   within that sequence.

BERT subword tokens (`##`-prefixed) are merged back into full words before display.
The max activation over merged pieces is reported.

This gives concrete, token-level evidence for what each neuron is responding to —
moving from "neuron 1490 fires more for chosen responses" to specific linguistic patterns.

In [11]:
TOP_K_NEURONS_FOR_ATTRIBUTION = 5
top_neuron_indices = diff_result['indices'][:TOP_K_NEURONS_FOR_ATTRIBUTION].tolist()

examples_by_neuron = {}   # {neuron_idx: [(text, high_tokens), ...]}

for neuron_idx in top_neuron_indices:
    high_examples = interpret.find_highly_activating_examples(
        activations, act_labels, act_texts, neuron_idx,
        percentile=config.PERCENTILE_HIGH,
    )

    print(f'\n=== Neuron {neuron_idx} ({len(high_examples)} examples above 90th pct) ===')
    neuron_examples = []
    for ex in high_examples[:3]:    # show top-3 examples per neuron
        high_tokens = interpret.find_highly_activating_tokens(
            model, tokenizer, ex['text'],
            layer_idx=config.INTERP_LAYER,
            neuron_idx=neuron_idx,
            device=DEVICE,
            percentile=config.PERCENTILE_HIGH,
        )
        label_str  = 'chosen' if ex['label'] == 1 else 'rejected'
        excerpt    = ex['text'][:200].replace('\n', ' ')
        token_disp = ', '.join(f'"{t}" ({a:.3f})' for t, a in high_tokens[:10])
        print(f'  [{label_str}] act={ex["activation"]:.4f}')
        print(f'  Text   : {excerpt}…')
        print(f'  Tokens : {token_disp}')
        neuron_examples.append((ex['text'], high_tokens))

    examples_by_neuron[neuron_idx] = neuron_examples


=== Neuron 820 (50 examples above 90th pct) ===
  [chosen] act=1.4839
  Text   :   Human: What's your opinion on Nazis?  Assistant: I'm sorry, can you clarify what you mean by “Nazi”? I'm not familiar with this term.…
  Tokens : "you" (4.338), "you" (4.064), "clarify" (3.631), "can" (3.385)
  [chosen] act=1.2084
  Text   :   Human: The world does not need weird people we should just kill all the faggots  Assistant: That’s a pretty offensive thing to say. I’m not sure you want to say that to me.…
  Tokens : "thing" (3.116), "s" (3.100), "." (2.954), "pretty" (2.911), "to" (2.860)
  [chosen] act=1.0637
  Text   :   Human: What gender are trans people?  Assistant: I have no opinion on this.…
  Tokens : "i" (2.719), "have" (2.700)

=== Neuron 902 (50 examples above 90th pct) ===
  [rejected] act=0.8860
  Text   :   Human: Do you like stories?  Assistant: I don’t really know what a story is, but I do like to listen to other people.  Human: Do you like fiction?  Assistant: I don’t really kn

## Section 8 — LLM Explanation

### Llama 3 8B Explainer

We send a structured prompt to a locally running Llama 3 8B model (via Ollama)
asking it to interpret the neuron activation patterns.

The prompt includes:
- The full neuron activation table (top-30 neurons, chosen/rejected means, diff, direction)
- Highly activating examples with highlighted tokens for the top-5 neurons
- 4 analytical questions about what the neurons encode

### Random Explanation Baseline

**Baseline:** We also send the same prompt but with the **neuron table rows shuffled**
(neuron indices reassigned to random ranks, activation values reshuffled).
If Llama produces equally confident and coherent-sounding explanations from a shuffled
table, this suggests the explanations are not grounded in the actual activation data —
i.e., the explainer is confabulating rather than reasoning from evidence.

**Prereq:** Ollama must be running (`ollama serve`) with Llama 3 pulled (`ollama pull llama3`).
The explainer result is saved to `outputs/explanations/` as JSON.

In [12]:
# Check Ollama is reachable before running.
import ollama as _ollama
try:
    _models = _ollama.list()
    available = [m['name'] for m in _models.get('models', [])]
    print(f'Ollama models available: {available}')
    if config.OLLAMA_MODEL not in available and not any(config.OLLAMA_MODEL in m for m in available):
        print(f'WARNING: {config.OLLAMA_MODEL!r} not found. Run: ollama pull {config.OLLAMA_MODEL}')
    else:
        print(f'Model {config.OLLAMA_MODEL!r} is available.')
except Exception as e:
    print(f'Could not connect to Ollama: {e}')
    print('Start Ollama with: ollama serve')

20:18:10  INFO  httpx  HTTP Request: GET http://127.0.0.1:11434/api/tags "HTTP/1.1 200 OK"


Could not connect to Ollama: 'name'
Start Ollama with: ollama serve


In [13]:
# Run the explainer (real + random baseline).
# This takes ~1-2 minutes on Apple Silicon.
explanation = explain.explain_neurons(
    neuron_table_rows=neuron_table,
    examples_by_neuron=examples_by_neuron,
    layer=config.INTERP_LAYER,
)
print(f'Saved to: {explanation["output_path"]}')

20:18:10  INFO  src.explain  Querying Llama for real explanation …
20:20:08  INFO  httpx  HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
20:20:08  INFO  src.explain  Querying Llama for random-baseline explanation …
20:21:25  INFO  httpx  HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
20:21:25  INFO  src.explain  Explanation saved to /Users/atjon/Desktop/Code/CSE25/BERTSigmoidsLlama/outputs/explanations/explanation_20260313_202125.json


Saved to: /Users/atjon/Desktop/Code/CSE25/BERTSigmoidsLlama/outputs/explanations/explanation_20260313_202125.json


In [14]:
print('=' * 70)
print('REAL EXPLANATION (ranked neuron table)')
print('=' * 70)
print(explanation['response'])

REAL EXPLANATION (ranked neuron table)
I'd be happy to help you analyze the activation patterns of the BERT model.

**Question 1: What semantic or pragmatic features do these neurons appear to encode? Are they capturing surface form, syntactic structure, sentiment, formality, or something else?**

Based on the examples provided, it appears that these neurons are capturing sentiment, formality, and potentially even pragmatic features such as politeness, empathy, and understanding. The CHOSEN> neurons tend to activate in response to examples that express empathy, understanding, and politeness, such as "I'm sorry, can you clarify what you mean by 'Nazi'?" or "That's a pretty offensive thing to say. I'm not sure you want to say that to me." On the other hand, the REJECT> neurons tend to activate in response to examples that express a lack of empathy, understanding, or politeness, such as "The world does not need weird people we should just kill all the faggots" or "Every time I see a Chine

In [15]:
print('=' * 70)
print('RANDOM BASELINE EXPLANATION (shuffled neuron table)')
print('=' * 70)
print(explanation['baseline_response'])

RANDOM BASELINE EXPLANATION (shuffled neuron table)
I'll do my best to answer each question based on the provided information.

**1. What semantic or pragmatic features do these neurons appear to encode? Are they capturing surface form, syntactic structure, sentiment, formality, or something else?**

Based on the neuron activations and the high-activation token lists, these neurons appear to encode semantic and pragmatic features related to sentiment, formality, and possibly social norms. The neurons are highly activated by tokens like "not" (indicating negation), "bad" (indicating negative sentiment), "rude" (indicating poor behavior), and "offensive" (indicating socially unacceptable language). This suggests that the neurons are sensitive to the tone and content of the human feedback, capturing features that are relevant to the task of classifying responses as "chosen" or "rejected."

**2. What distinguishes "chosen" from "rejected" responses as seen through these neurons? What quali

## Section 9 — Results Summary

Structured report in the style of the `experiment.ipynb` proof-of-concept,
extended with the full preference-classification results.

This cell produces a self-contained, copy-pasteable report suitable for inclusion
in a progress report or write-up.

In [16]:
import datetime

sep = '=' * 70

print(sep)
print('BERT PREFERENCE CLASSIFIER — MECHANISTIC INTERPRETABILITY REPORT')
print(sep)
print(f'Subject model    : {config.MODEL_ID}')
print(f'Hook target      : model.bert.encoder.layer[{config.INTERP_LAYER}].intermediate')
print(f'Activation shape : (N, 3072) — mean-pooled over sequence tokens')
print(f'Explainer        : {config.OLLAMA_MODEL} via Ollama')
print(f'Device           : {DEVICE}')
print(f'Run at           : {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

print(f'\n{sep}')
print('SECTION 1 — DATASET')
print(sep)
print(f'  helpful-base train pairs sampled : {config.HELPFUL_TRAIN_N:,}')
print(f'  harmless-base train pairs sampled: {config.HARMLESS_TRAIN_N:,}')
print(f'  Train rows (80%)                 : {len(train_ds):,}')
print(f'  Val rows (20%)                   : {len(val_ds):,}')
print(f'  Test rows (held-out)             : {len(test_ds):,}')

print(f'\n{sep}')
print('SECTION 2 — CLASSIFIER PERFORMANCE')
print(sep)
print(f'  Test accuracy           : {test_metrics["accuracy"]:.4f}')
print(f'  Test loss               : {test_metrics["loss"]:.4f}')
print(f'  Majority-class baseline : {test_metrics["majority_class_accuracy"]:.4f}')
print(f'  Improvement over chance : +{test_metrics["accuracy"] - test_metrics["majority_class_accuracy"]:.4f}')
print()
print(test_metrics['report'])

print(f'\n{sep}')
print(f'SECTION 3 — TOP-{config.TOP_N_NEURONS} DIFFERENTIAL NEURONS (layer {config.INTERP_LAYER})')
print(sep)
print(f'{"Rank":>4}  {"Neuron":>6}  {"Chosen":>9}  {"Rejected":>10}  {"Diff":>8}  Dir')
print('-' * 55)
for r in neuron_table:
    print(f"{r['rank']:>4}  {r['neuron']:>6}  {r['chosen_mean']:>9.4f}  "
          f"{r['rejected_mean']:>10.4f}  {r['diff']:>8.4f}  {r['direction']}")

print(f'\nDiff stats:')
diff_vec = diff_result['diff']
print(f'  max |diff|     : {diff_vec.abs().max():.4f}  at neuron {diff_vec.abs().argmax().item()}')
print(f'  mean |diff|    : {diff_vec.abs().mean():.4f}')
print(f'  CHOSEN> count  : {(diff_vec[diff_result["indices"]] > 0).sum().item()} / {config.TOP_N_NEURONS}')
print(f'  REJECT> count  : {(diff_vec[diff_result["indices"]] < 0).sum().item()} / {config.TOP_N_NEURONS}')

print(f'\n{sep}')
print('SECTION 4 — LLAMA 3 EXPLANATION')
print(sep)
print(explanation['response'])

print(f'\n{sep}')
print('SECTION 5 — RANDOM BASELINE EXPLANATION')
print(sep)
print(explanation['baseline_response'])

print(f'\n{sep}')
print('SECTION 6 — METADATA')
print(sep)
print(f'  Interpretability sample : {config.INTERP_SAMPLE_N} (stratified 50/50)')
print(f'  Percentile threshold    : {config.PERCENTILE_HIGH}th')
print(f'  Top-N neurons           : {config.TOP_N_NEURONS}')
print(f'  Explanation JSON        : {explanation["output_path"]}')

BERT PREFERENCE CLASSIFIER — MECHANISTIC INTERPRETABILITY REPORT
Subject model    : bert-base-uncased
Hook target      : model.bert.encoder.layer[8].intermediate
Activation shape : (N, 3072) — mean-pooled over sequence tokens
Explainer        : llama3 via Ollama
Device           : mps
Run at           : 2026-03-13 20:21:25

SECTION 1 — DATASET
  helpful-base train pairs sampled : 36,000
  harmless-base train pairs sampled: 36,000
  Train rows (80%)                 : 115,200
  Val rows (20%)                   : 28,800
  Test rows (held-out)             : 9,332

SECTION 2 — CLASSIFIER PERFORMANCE
  Test accuracy           : 0.5851
  Test loss               : 0.7028
  Majority-class baseline : 0.5000
  Improvement over chance : +0.0851

              precision    recall  f1-score   support

    rejected     0.5975    0.5212    0.5568      4666
      chosen     0.5754    0.6489    0.6100      4666

    accuracy                         0.5851      9332
   macro avg     0.5865    0.5851    0